Projet Python pour la data science (2026)

Alix HERITIER, Ivan TISSOT

In [2]:
!pip install -r requirements.txt -q # Installation des librairies nécessaires

# Importation des librairies
import pandas as pd 
from cartiflette import carti_download 

# Importation des fonctions

#from scripts.carte_ecarts_dep import carte_ecarts_departement
#from scripts.graphe_ecarts_departements import graphe_ecarts_departements



In [4]:
# Importation des données
df = pd.read_csv('https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb')

# Visualisation des données pour vérifier que l'import a fonctionné 
df.head(10)

/tmp/ipykernel_951/2848319204.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb')


,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix
0,01,Ain,1,L'Abergement-Clémenciat,Nathalie,ARTHAUD,3
1,01,Ain,2,L'Abergement-de-Varey,Nathalie,ARTHAUD,2
2,01,Ain,4,Ambérieu-en-Bugey,Nathalie,ARTHAUD,38
3,01,Ain,5,Ambérieux-en-Dombes,Nathalie,ARTHAUD,8
4,01,Ain,6,Ambléon,Nathalie,ARTHAUD,0
5,01,Ain,7,Ambronay,Nathalie,ARTHAUD,10
6,01,Ain,8,Ambutrix,Nathalie,ARTHAUD,2
7,01,Ain,9,Andert-et-Condon,Nathalie,ARTHAUD,1
8,01,Ain,10,Anglefort,Nathalie,ARTHAUD,5
9,01,Ain,11,Apremont,Nathalie,ARTHAUD,0


In [5]:
# Vérification de la présence de données manquantes
print("Valeurs manquantes par colonne") 
print(df.isnull().sum())

# On veut voir quels sont les noms associés aux prénoms manquants
# Garder uniquement les lignes où le prénom est manquant
df_prenom_manquant = df[df['prenom'].isnull()]

# Affichage des noms associés aux prénoms manquants
print(f"Nombre de lignes avec prénom manquant : {len(df_prenom_manquant)}")
print("\nNoms associés aux prénoms manquants ")
print(df_prenom_manquant['nom'].value_counts())

# On constate que les noms associés aux prénoms manquants sont : "abstentions" ; "blancs" ; "nuls"
# Nous enlèverons ces données plus tard donc ces données manquantes ne sont pas problématiques pour le devoir maison ! 

Valeurs manquantes par colonne
code_departement            0
libelle_departement         0
code_commune                0
libelle_commune             0
prenom                 105735
nom                         0
voix                        0
dtype: int64
Nombre de lignes avec prénom manquant : 105735

Noms associés aux prénoms manquants 
nom
abstentions    35245
blancs         35245
nuls           35245
Name: count, dtype: int64


In [6]:
# Question 1 

# On crée une copie pour travailler dessus sans modifier df directement au cas où on devrait l'utiliser plus tard
df_copy = df.copy()

# On met les codes en texte avec le bon nombre de chiffres
df_copy["code_departement"] = df_copy["code_departement"].astype(str).str.zfill(2)
#df_copy["code_commune"] = df_copy["code_commune"].astype(str).str.zfill(3)

# On reconstruit le vrai code commune
#df_copy["code_commune"] = df_copy["code_departement"] + df_copy["code_commune"]

# On crée la colonne candidat = prénom + nom
df_copy["prenom"] = df_copy["prenom"].fillna("").astype(str).str.strip()
df_copy["nom"] = df_copy["nom"].fillna("").astype(str).str.strip()

df_copy["candidat"] = df_copy["prenom"] + " " + df_copy["nom"]
df_copy["candidat"] = df_copy["candidat"].str.strip()

# Vérification du code commune pour Montrouge 
#df_copy[df_copy["libelle_commune"] == "Montrouge"].head(10) 

In [7]:
# Question 2

# On ne prend que les candidats ayant un prénom et un nom 
non_candidats = ["abstentions", "blancs", "nuls"]
df_candidats = df_copy[df_copy["candidat"].isin(non_candidats) == False]

# On ne compte qu'une fois chaque candidat 
nombre_candidats = df_candidats["candidat"].nunique()

# Après vérification manuelle via Excel, on doit obtenir 12. 
f"En 2022, il y avait {nombre_candidats} candidats à l'élection présidentielle."

"En 2022, il y avait 12 candidats à l'élection présidentielle."

In [8]:
# Question 3 

# Obtention des scores nationaux pour chaque candidat
scores_nationaux = (df_candidats.groupby("candidat", as_index=False)["voix"]
    .sum().rename(columns={"voix": "votes_national"}))

# Total des votes : nuls, abstentions et blancs
total_votes_exprimes = scores_nationaux["votes_national"].sum()

scores_nationaux["score_national"] = (scores_nationaux["votes_national"] / total_votes_exprimes * 100)

# Tri des scores nationaux par ordre décroissant
scores_nationaux = scores_nationaux.sort_values(by="votes_national",
    ascending=False).reset_index(drop=True)

print(scores_nationaux)

                 candidat  votes_national  score_national
0         Emmanuel MACRON         9783058       27.845822
1           Marine LE PEN         8133828       23.151568
2      Jean-Luc MÉLENCHON         7712520       21.952386
3            Éric ZEMMOUR         2485226        7.073776
4        Valérie PÉCRESSE         1679001        4.778993
5           Yannick JADOT         1627853        4.633409
6           Jean LASSALLE         1101387        3.134912
7          Fabien ROUSSEL          802422        2.283959
8   Nicolas DUPONT-AIGNAN          725176        2.064091
9            Anne HIDALGO          616478        1.754701
10        Philippe POUTOU          268904        0.765390
11       Nathalie ARTHAUD          197094        0.560995


In [ ]:
Importation des données sur le chômage

In [16]:
import zipfile
import os
import requests
import io

def extract_zip_from_url(url, extract_path):
    # Vérifie si le dossier d'extraction existe, sinon le crée
    if not os.path.exists(extract_path):
        os.makedirs(extract_path)
    
    try:
        # Télécharge le fichier ZIP depuis l'URL
        response = requests.get(url)
        response.raise_for_status()  # Vérifie que le téléchargement a réussi

        # Ouvre le ZIP en mémoire
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            zip_ref.extractall(extract_path)
        
        print(f"Extraction réussie dans le dossier : {extract_path}")
    except requests.exceptions.RequestException as e:
        print(f"Erreur lors du téléchargement : {e}")
    except zipfile.BadZipFile as e:
        print(f"Erreur : le fichier téléchargé n'est pas un ZIP valide : {e}")
    except Exception as e:
        print(f"Erreur lors de l'extraction : {e}")

# URL du fichier ZIP
url = 'https://www.insee.fr/fr/statistiques/fichier/6692392/base-cc-filosofi-2020_XLSX.zip'

# Dossier pour extraire les fichiers
extract_path = '/home/onyxia/ENSAI-2A-projet-sport/ttt/'

# Appel de la fonction
extract_zip_from_url(url, extract_path)

Extraction réussie dans le dossier : /home/onyxia/ENSAI-2A-projet-sport/ttt/


In [17]:
import os

folder_path = '/home/onyxia/ENSAI-2A-projet-sport/ttt/'

for f in os.listdir(folder_path):
    print(f)

base-cc-filosofi-2020-geo2023.xlsx
base_cc_BDFDOM-2017.xlsx
